
# Microsoft Foundry Local 
## Advanced Function Calling with Foundry Local

<img src="https://github.com/retkowsky/foundry-local/blob/main/foundrylocal.jpg?raw=true">

This notebook demonstrates **end-to-end function calling** with Foundry Local, including:

- Defining a rich set of tools (weather, stock prices, unit conversion, web search, email)
- Parsing the model's tool-call responses
- Executing mock tool functions locally
- Feeding results back to the model for a final answer
- Handling **parallel** and **chained** function calls

> **Docs:** [Foundry Local Documentation](https://learn.microsoft.com/azure/ai-foundry/foundry-local/)  
> **SDK Reference:** [Python SDK Reference](https://learn.microsoft.com/azure/ai-foundry/foundry-local/reference/reference-sdk?pivots=programming-language-python)

## Author

| Field | Details |
| --- | --- |
| Name | Serge Retkowsky |
| Email | serge.retkowsky@microsoft.com |
| LinkedIn | https://www.linkedin.com/in/serger/ |
| Medium publications | https://medium.com/@sergems18/ |

In [10]:
import datetime
import GPUtil
import json
import openai
import os
import pandas as pd
import platform
import psutil
import random
import sys

from datetime import datetime, timedelta
from foundry_local import FoundryLocalManager

In [2]:
print(f"Python version: {sys.version}")

Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]


In [3]:
print(f"Today is {datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

Today is 25-Feb-2026 15:48:31


In [4]:
print(f"💻 OS: {platform.system()} {platform.release()}")
print(f"- CPU: {platform.processor()}")
print(
    f"- CPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical"
)

ram = psutil.virtual_memory()
print(f"- RAM total:     {ram.total / (1024**3):.1f} GB")
print(f"- RAM available: {ram.available / (1024**3):.1f} GB")
print(f"- RAM used:      {ram.percent}%")

for part in psutil.disk_partitions():
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"\n💾 Disk [{part.device}] mounted on {part.mountpoint}")
        print(f"- Total: {usage.total / (1024**3):.1f} GB")
        print(f"- Used:  {usage.used / (1024**3):.1f} GB ({usage.percent}%)")
        print(f"- Free:  {usage.free / (1024**3):.1f} GB")
    except PermissionError:
        pass

gpus = GPUtil.getGPUs()

if not gpus:
    print("No GPU detected.")
else:
    for i, gpu in enumerate(gpus):
        print(f"\n🎮 GPU {i} — {gpu.name}")
        print(f"- VRAM Total : {gpu.memoryTotal:,.0f} MB")
        print(
            f"- VRAM Used  : {gpu.memoryUsed:,.0f} MB ({gpu.memoryUtil * 100:.0f}%)"
        )
        print(f"- VRAM Free  : {gpu.memoryFree:,.0f} MB")
        print(f"- GPU Load   : {gpu.load * 100:.0f}%")
        print(f"- Temperature: {gpu.temperature} °C")

💻 OS: Windows 11
- CPU: Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
- CPU cores: 8 physical, 16 logical
- RAM total:     63.7 GB
- RAM available: 40.2 GB
- RAM used:      36.9%

💾 Disk [C:\] mounted on C:\
- Total: 951.6 GB
- Used:  869.6 GB (91.4%)
- Free:  82.0 GB

🎮 GPU 0 — NVIDIA T1200 Laptop GPU
- VRAM Total : 4,096 MB
- VRAM Used  : 1,658 MB (40%)
- VRAM Free  : 2,278 MB
- GPU Load   : 33%
- Temperature: 66.0 °C


In [5]:
manager = FoundryLocalManager()
manager.start_service()

print(f"Service running : {manager.is_service_running()}")
print(f"Service URI     : {manager.service_uri}")
print(f"Endpoint (v1)   : {manager.endpoint}")
print(f"Cache location  : {manager.get_cache_location()}")

Service running : True
Service URI     : http://127.0.0.1:55311
Endpoint (v1)   : http://127.0.0.1:55311/v1
Cache location  : C:\models


In [6]:
catalog = manager.list_catalog_models()
print(f"Total model in catalog = {len(catalog)}")

Total model in catalog = 80


In [7]:
for idx, model in enumerate(catalog, start=1):
    print(f"{idx}: {model.id} - {model.device_type}")

1: Phi-4-cuda-gpu:1 - GPU
2: phi-4-openvino-gpu:1 - GPU
3: Phi-4-generic-gpu:1 - GPU
4: Phi-4-generic-cpu:1 - CPU
5: Phi-3.5-mini-instruct-cuda-gpu:1 - GPU
6: Phi-3.5-mini-instruct-openvino-gpu:1 - GPU
7: Phi-3.5-mini-instruct-generic-gpu:1 - GPU
8: Phi-3.5-mini-instruct-generic-cpu:1 - CPU
9: Phi-3-mini-128k-instruct-cuda-gpu:1 - GPU
10: Phi-3-mini-128k-instruct-openvino-gpu:1 - GPU
11: Phi-3-mini-128k-instruct-generic-gpu:1 - GPU
12: Phi-3-mini-128k-instruct-generic-cpu:2 - CPU
13: Phi-3-mini-4k-instruct-cuda-gpu:1 - GPU
14: Phi-3-mini-4k-instruct-openvino-gpu:1 - GPU
15: Phi-3-mini-4k-instruct-generic-gpu:1 - GPU
16: Phi-3-mini-4k-instruct-generic-cpu:2 - CPU
17: mistralai-Mistral-7B-Instruct-v0-2-cuda-gpu:1 - GPU
18: Mistral-7B-Instruct-v0-2-openvino-gpu:1 - GPU
19: mistralai-Mistral-7B-Instruct-v0-2-generic-gpu:1 - GPU
20: mistralai-Mistral-7B-Instruct-v0-2-generic-cpu:2 - CPU
21: deepseek-r1-distill-qwen-14b-cuda-gpu:3 - GPU
22: DeepSeek-R1-Distill-Qwen-14B-openvino-gpu:1 - GPU
2

In [8]:
def models_to_df(models):
    """Convert a list of FoundryModelInfo objects into a clean DataFrame."""
    return pd.DataFrame([{
        "alias": m.alias,
        "id": m.id,
        "device": m.device_type.value,
        "provider": m.execution_provider,
        "size_mb": m.file_size_mb,
        "tools": m.supports_tool_calling,
        "license": m.license,
        "task": m.task,
    } for m in models])

In [11]:
catalog = manager.list_catalog_models()
print(f"Total model variants in catalog = {len(catalog)}")

df_catalog = models_to_df(catalog)
df_catalog

Total model variants in catalog = 80


,alias,id,device,provider,size_mb,tools,license,task
0,phi-4,Phi-4-cuda-gpu:1,GPU,CUDAExecutionProvider,8570,False,MIT,chat-completion
1,phi-4,phi-4-openvino-gpu:1,GPU,OpenVINOExecutionProvider,9046,False,MIT,chat-completion
2,phi-4,Phi-4-generic-gpu:1,GPU,WebGpuExecutionProvider,8570,False,MIT,chat-completion
3,phi-4,Phi-4-generic-cpu:1,CPU,CPUExecutionProvider,10403,False,MIT,chat-completion
4,phi-3.5-mini,Phi-3.5-mini-instruct-cuda-gpu:1,GPU,CUDAExecutionProvider,2181,False,MIT,chat-completion
...,...,...,...,...,...,...,...,...
75,qwen2.5-7b,qwen2.5-7b-instruct-generic-cpu:4,CPU,CPUExecutionProvider,6307,True,apache-2.0,chat-completion
76,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-cuda-gpu:2,GPU,CUDAExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
77,whisper-large-v3-turbo,openai-whisper-large-v3-turbo-generic-cpu:2,CPU,CPUExecutionProvider,9000,False,apache-2.0,automatic-speech-recognition
78,gpt-oss-20b,gpt-oss-20b-generic-cpu:1,CPU,CPUExecutionProvider,12552,False,MIT,chat-completion


## Let's use "Phi-4" model

- Privacy and Control: Your data stays entirely on your device — a critical advantage for sensitive domains like healthcare, finance, or education, where data confidentiality is non-negotiable.
- Cost Efficiency: Forget about API fees and rate limits. Once the model is downloaded, you can run inference as often as you like at zero cost.
- Speed and Reliability: With no network latency or dependence on external services, your AI applications respond instantly — and keep working even offline.
- Learning and Experimentation: Enjoy complete freedom to tweak model parameters, craft custom prompts, and explore fine-tuning without any platform restrictions.

In [12]:
MODEL_ALIAS = "Phi-4"

In [13]:
# Download first (this may take a few minutes depending on model size)
manager.download_model("Phi-4")

FoundryModelInfo(alias=phi-4, id=Phi-4-cuda-gpu:1, execution_provider=CUDAExecutionProvider, device_type=GPU, file_size=8570 MB, license=MIT)

In [14]:
os.listdir(os.path.join(manager.get_cache_location(), "Microsoft"))

['Phi-4-cuda-gpu-1']

In [19]:
cached = manager.list_cached_models()
print(f"Cached models = {len(cached)}")

print(cached)

Cached models = 1
[FoundryModelInfo(alias=phi-4, id=Phi-4-cuda-gpu:1, execution_provider=CUDAExecutionProvider, device_type=GPU, file_size=8570 MB, license=MIT)]


In [20]:
manager = FoundryLocalManager(MODEL_ALIAS)
model_id = manager.get_model_info(MODEL_ALIAS).id

In [21]:
info = manager.get_model_info("Phi-4")
print(f"Running on: {info.execution_provider}")
print(f"Device:     {info.device_type}")

Running on: CUDAExecutionProvider
Device:     GPU


In [22]:
client = openai.OpenAI(
    base_url=manager.endpoint,
    api_key=manager.api_key,
)

print(f"Connected model: {model_id}")
print(f"endpoint: {manager.endpoint}")

Connected model: Phi-4-cuda-gpu:1
endpoint: http://127.0.0.1:55311/v1


## Define the Tools catalogue

In [23]:
# OpenAI-compatible tool definitions for the API
TOOLS_OPENAI = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather and 3-day forecast for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Paris' or 'Tokyo'"
                },
                "units": {
                    "type": "string",
                    "description":
                    "Temperature unit: 'celsius' or 'fahrenheit'",
                    "enum": ["celsius", "fahrenheit"]
                }
            },
            "required": ["city"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "get_stock_price",
        "description":
        "Get the latest stock price, daily change, and 52-week range for a ticker symbol.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol, e.g. 'AAPL', 'MSFT'"
                }
            },
            "required": ["ticker"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "convert_currency",
        "description":
        "Convert an amount from one currency to another using current exchange rates.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "The amount of money to convert"
                },
                "from_currency": {
                    "type": "string",
                    "description": "ISO 4217 code, e.g. 'USD'"
                },
                "to_currency": {
                    "type": "string",
                    "description": "ISO 4217 code, e.g. 'EUR'"
                }
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "search_web",
        "description":
        "Search the web and return top results with title, snippet, and URL.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query string"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Number of results (1-10)"
                }
            },
            "required": ["query"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "send_email",
        "description": "Send an email to a recipient with subject and body.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "Recipient email address"
                },
                "subject": {
                    "type": "string",
                    "description": "Email subject line"
                },
                "body": {
                    "type": "string",
                    "description": "Email body text"
                }
            },
            "required": ["to", "subject", "body"]
        }
    }
}]

# Simple tool list for the system prompt (model-friendly format)
TOOLS_SIMPLE = [{
    "name": t["function"]["name"],
    "description": t["function"]["description"],
    "parameters": t["function"]["parameters"]["properties"]
} for t in TOOLS_OPENAI]

print(f"Registered {len(TOOLS_OPENAI)} tools:")

for tool in TOOLS_OPENAI:
    f = tool['function']
    print(f"- {f['name']}: {f['description']}")

Registered 5 tools:
- get_weather: Get current weather and 3-day forecast for a city.
- get_stock_price: Get the latest stock price, daily change, and 52-week range for a ticker symbol.
- convert_currency: Convert an amount from one currency to another using current exchange rates.
- search_web: Search the web and return top results with title, snippet, and URL.
- send_email: Send an email to a recipient with subject and body.


## System Prompt for Tool Calling

Small local models like phi-4-mini **need an explicit system prompt** that:
1. Tells the model what tools are available (as a JSON string).
2. Instructs it to respond with the `functools[...]` format.
3. Tells it to call one or more tools in parallel when needed.

Without this, the model will just answer conversationally and ignore the tools.

In [24]:
TOOL_LIST_JSON = json.dumps(TOOLS_SIMPLE, indent=5)

SYSTEM_PROMPT = f"""You are a helpful assistant with access to the following tools:

{TOOL_LIST_JSON}

When the user's request can be answered using one or more of these tools, you MUST respond ONLY with a function call in this exact format:

functools[{{"name": "tool_name", "arguments": {{...}}}}]

For multiple tools, use a JSON array:

functools[{{"name": "tool1", "arguments": {{...}}}}, {{"name": "tool2", "arguments": {{...}}}}]

Rules:
- ALWAYS use a tool when the request matches a tool's purpose.
- Do NOT add any text before or after the functools[...] output.
- Use correct parameter names and types as defined above.
- If multiple tools are needed, call them all in a single functools[] response.
- Only answer directly (without tools) if no tool is relevant to the question.
"""

In [25]:
print(SYSTEM_PROMPT)

You are a helpful assistant with access to the following tools:

[
     {
          "name": "get_weather",
          "description": "Get current weather and 3-day forecast for a city.",
          "parameters": {
               "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Paris' or 'Tokyo'"
               },
               "units": {
                    "type": "string",
                    "description": "Temperature unit: 'celsius' or 'fahrenheit'",
                    "enum": [
                         "celsius",
                         "fahrenheit"
                    ]
               }
          }
     },
     {
          "name": "get_stock_price",
          "description": "Get the latest stock price, daily change, and 52-week range for a ticker symbol.",
          "parameters": {
               "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol, e.g. 'AAPL', 'MSFT'"
    

## Functions

These functions simulate real API calls. **In production you would replace them with actual service calls.**

In [26]:
def get_weather(city: str, units: str = "celsius") -> dict:
    """Simulate a weather API response."""
    base_temp = {
        "celsius": random.randint(5, 30),
        "fahrenheit": random.randint(40, 86)
    }
    temp = base_temp.get(units, base_temp["celsius"])
    conditions = random.choice(
        ["Sunny", "Partly Cloudy", "Overcast", "Light Rain", "Thunderstorms"])

    forecast = []
    for i in range(1, 4):
        day = (datetime.now() + timedelta(days=i)).strftime("%A %b %d")
        forecast.append({
            "day":
            day,
            "high":
            temp + random.randint(-3, 5),
            "low":
            temp - random.randint(2, 8),
            "conditions":
            random.choice(["Sunny", "Cloudy", "Rain"])
        })

    return {
        "city": city,
        "temperature": temp,
        "units": units,
        "conditions": conditions,
        "humidity": random.randint(30, 90),
        "wind_kph": random.randint(5, 40),
        "forecast": forecast
    }


def get_stock_price(ticker: str) -> dict:
    """Simulate a stock API response."""
    prices = {
        "AAPL": 234.56,
        "MSFT": 452.30,
        "GOOGL": 178.90,
        "AMZN": 198.45,
        "NVDA": 135.20,
        "TSLA": 265.80
    }
    price = prices.get(ticker.upper(), round(random.uniform(20, 500), 2))
    change_pct = round(random.uniform(-4, 4), 2)

    return {
        "ticker": ticker.upper(),
        "price": price,
        "change_pct": change_pct,
        "change_direction": "up" if change_pct > 0 else "down",
        "52w_high": round(price * random.uniform(1.1, 1.4), 2),
        "52w_low": round(price * random.uniform(0.5, 0.85), 2),
        "volume": f"{random.randint(10, 120)}M",
        "market_cap": f"${random.randint(100, 3500)}B"
    }


def convert_currency(amount: float, from_currency: str,
                     to_currency: str) -> dict:
    """Simulate a currency conversion API."""
    rates = {
        ("USD", "EUR"): 0.92,
        ("EUR", "USD"): 1.09,
        ("USD", "GBP"): 0.79,
        ("GBP", "USD"): 1.27,
        ("USD", "JPY"): 149.50,
        ("JPY", "USD"): 0.0067,
        ("EUR", "GBP"): 0.86,
        ("GBP", "EUR"): 1.16,
        ("EUR", "JPY"): 162.80,
        ("USD", "CNY"): 7.24,
    }
    key = (from_currency.upper(), to_currency.upper())
    rate = rates.get(key, round(random.uniform(0.5, 150), 4))

    return {
        "from": from_currency.upper(),
        "to": to_currency.upper(),
        "amount": amount,
        "rate": rate,
        "converted": round(amount * rate, 2),
        "timestamp": datetime.now().isoformat()
    }


def search_web(query: str, max_results: int = 3) -> dict:
    """Simulate a web search API."""
    results = [{
        "title": f"Result {i+1} for '{query}'",
        "snippet":
        f"This is a relevant excerpt about {query} from source {i+1}...",
        "url": f"https://example.com/result{i+1}"
    } for i in range(min(max_results, 5))]
    return {
        "query": query,
        "total_results": random.randint(1000, 50000),
        "results": results
    }


def send_email(to: str, subject: str, body: str) -> dict:
    """Simulate sending an email."""
    return {
        "status": "sent",
        "to": to,
        "subject": subject,
        "message_id": f"msg-{random.randint(100000, 999999)}",
        "timestamp": datetime.now().isoformat()
    }


# Registry: maps tool name → Python callable
TOOL_REGISTRY = {
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "convert_currency": convert_currency,
    "search_web": search_web,
    "send_email": send_email,
}

## Tool-Call Parser & Executor

Foundry Local returns tool calls inside the content as a JSON string prefixed by `functools`.
The helper below:
1. Collects the streamed response.
2. Detects if the model wants to call tools.
3. Parses the JSON, calls each mock function, and returns the results.

In [27]:
def collect_stream(stream) -> str:
    """Collect all chunks from a streaming response into a single string."""
    parts = []
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            parts.append(delta)
    return "".join(parts)


def parse_and_execute_tools(raw_response: str) -> list[dict] | None:
    """
    If the model returned tool calls (prefixed with 'functools'),
    parse them and execute each one.
    Returns a list of {tool_name, arguments, result} dicts, or None.
    """
    if not raw_response.strip().startswith("functools"):
        return None  # Plain text answer, no tool calls

    json_str = raw_response.strip().removeprefix("functools").strip()
    tool_calls = json.loads(json_str)

    # Normalise to a list (model may return a single dict)

    if isinstance(tool_calls, dict):
        tool_calls = [tool_calls]

    results = []
    for tc in tool_calls:
        name = tc["name"]
        args = tc.get("arguments", {})
        func = TOOL_REGISTRY.get(name)
        if func is None:
            result = {"error": f"Unknown tool: {name}"}
        else:
            result = func(**args)
        results.append({"tool": name, "arguments": args, "result": result})
        print(f"  🔧 {name}({args})")
        print(f"     ↳ {json.dumps(result, indent=2)[:200]}...\n")

    return results

## Full Agent Loop

The `agent_ask` function implements a full **tool-use loop**:
1. Send the **system prompt** (with tool definitions) + user message.
2. If the model returns `functools[...]` → parse and execute tools.
3. Append tool results to the conversation and call the model again for a natural-language summary.
4. Return the final answer.

> **Key**: The system prompt is essential — without it, small models like phi-4-mini  
> will answer conversationally instead of generating tool calls.

In [28]:
def chat(user_message: str, verbose: bool = True) -> str:
    """
    Send a user message through a full tool-calling agent loop.
    Returns the final natural-language answer.
    """
    if verbose:
        print(f"\n{'-'*100}")
        print(f"👤 USER: {user_message}")
        print(f"{'-'*100}\n")

    # Build messages WITH the system prompt — this is the key!
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        },
    ]

    # --- Step 1: first call (may trigger tool calls) ---
    if verbose:
        print("1️⃣ Step 1: Calling model")

    stream = client.chat.completions.create(
        model=model_id,
        messages=messages,
        tools=TOOLS_OPENAI,
        temperature=0.0,
        max_tokens=4096,
        top_p=1.0,
        stream=True,
    )
    raw = collect_stream(stream)

    if verbose:
        print(
            f"Raw response: {raw[:2000]}{'...' if len(raw) > 2000 else ''}\n")

    # --- Step 2: parse & execute tools if needed ---
    tool_results = parse_and_execute_tools(raw)

    if tool_results is None:
        # Model answered directly — no tools needed
        if verbose:
            print("✅ Model answered directly (no tool calls).\n")
        return raw

    if tool_results:
        print("✅ Calling tool calls.\n")

    # --- Step 3: feed results back and get final answer ---
    if verbose:
        print("2️⃣ Step 2: Feeding tool results back to model...\n")

    tool_results_text = json.dumps(tool_results, indent=2, default=str)
    messages.append({"role": "assistant", "content": raw})
    messages.append({
        "role":
        "user",
        "content":
        (f"Here are the tool results:\n{tool_results_text}\n\n"
         "Please provide a helpful, natural-language summary of these results "
         "for the user. Do NOT call any tools.")
    })

    stream = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=0.7,
        max_tokens=4096,
        top_p=1.0,
        stream=True,
    )
    final_answer = collect_stream(stream)

    if verbose:
        print(f"🏆 ASSISTANT:\n{final_answer}\n")

    return final_answer

## Examples

### Single tool call: Weather

In [29]:
_ = chat("What's the weather like in Paris right now? Use celsius.")


----------------------------------------------------------------------------------------------------
👤 USER: What's the weather like in Paris right now? Use celsius.
----------------------------------------------------------------------------------------------------

1️⃣ Step 1: Calling model
Raw response: functools[{"name": "get_weather", "arguments": {"city": "Paris", "units": "celsius"}}]

  🔧 get_weather({'city': 'Paris', 'units': 'celsius'})
     ↳ {
  "city": "Paris",
  "temperature": 26,
  "units": "celsius",
  "conditions": "Partly Cloudy",
  "humidity": 38,
  "wind_kph": 19,
  "forecast": [
    {
      "day": "Thursday Feb 26",
      "high":...

✅ Calling tool calls.

2️⃣ Step 2: Feeding tool results back to model...

🏆 ASSISTANT:
The current weather in Paris is 26°C with partly cloudy conditions. The humidity is at 38%, and the wind is blowing at 19 km/h. Looking ahead, the forecast for the next three days is as follows:

- **Thursday, February 26**: Expect rain with a high 

### Single tool call: Stock price
This is a simulated example.

In [30]:
_ = chat("How is NVDA stock doing today? Give me the price and 52-week range.")


----------------------------------------------------------------------------------------------------
👤 USER: How is NVDA stock doing today? Give me the price and 52-week range.
----------------------------------------------------------------------------------------------------

1️⃣ Step 1: Calling model
Raw response: functools[{"name": "get_stock_price", "arguments": {"ticker": "NVDA"}}]

  🔧 get_stock_price({'ticker': 'NVDA'})
     ↳ {
  "ticker": "NVDA",
  "price": 135.2,
  "change_pct": 3.58,
  "change_direction": "up",
  "52w_high": 162.73,
  "52w_low": 74.1,
  "volume": "106M",
  "market_cap": "$3392B"
}...

✅ Calling tool calls.

2️⃣ Step 2: Feeding tool results back to model...

🏆 ASSISTANT:
As of today, NVIDIA (NVDA) stock is priced at $135.20, which represents a 3.58% increase, indicating an upward movement. The stock's 52-week range shows a high of $162.73 and a low of $74.10. The trading volume for the day is 106 million shares, and NVIDIA's market capitalization is approxi

### Parallel tool calls: Multi-stock comparison

Ask the model to compare several stocks — it should call `get_stock_price` multiple times. This is a simulated example.

In [31]:
_ = chat(
    "Compare the current stock prices of AAPL, MSFT, and GOOGL. "
    "Which one had the best day?"
)


----------------------------------------------------------------------------------------------------
👤 USER: Compare the current stock prices of AAPL, MSFT, and GOOGL. Which one had the best day?
----------------------------------------------------------------------------------------------------

1️⃣ Step 1: Calling model
Raw response: functools[{"name": "get_stock_price", "arguments": {"ticker": "AAPL"}}, {"name": "get_stock_price", "arguments": {"ticker": "MSFT"}}, {"name": "get_stock_price", "arguments": {"ticker": "GOOGL"}}]

  🔧 get_stock_price({'ticker': 'AAPL'})
     ↳ {
  "ticker": "AAPL",
  "price": 234.56,
  "change_pct": 3.29,
  "change_direction": "up",
  "52w_high": 322.9,
  "52w_low": 166.81,
  "volume": "18M",
  "market_cap": "$3426B"
}...

  🔧 get_stock_price({'ticker': 'MSFT'})
     ↳ {
  "ticker": "MSFT",
  "price": 452.3,
  "change_pct": 1.69,
  "change_direction": "up",
  "52w_high": 568.89,
  "52w_low": 235.48,
  "volume": "70M",
  "market_cap": "$1131B"
}...

  🔧

### No tool needed (plain question)

The model should answer directly without invoking any tools.

In [33]:
_ = chat("What is the capital of France?")


----------------------------------------------------------------------------------------------------
👤 USER: What is the capital of France?
----------------------------------------------------------------------------------------------------

1️⃣ Step 1: Calling model
Raw response: The capital of France is Paris.

✅ Model answered directly (no tool calls).



> Go to the next notebook